# LLM07: FlashAttention — GPU-Efficient Attention

## Lab Overview

Standard self-attention has $O(N^2)$ HBM (High-Bandwidth Memory) access, making it the main bottleneck for long sequences. **FlashAttention** reduces memory access by leveraging GPU memory hierarchy: keeping intermediate results in fast on-chip SRAM (shared memory / registers) and minimizing round-trips to slow off-chip HBM (global memory).

This lab walks through the GPU memory model, tiling, online softmax, and the FlashAttention algorithm, with runnable code so you can benchmark standard vs. flash attention yourself.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

Understand why FlashAttention is faster (I/O analysis, not just FLOPs), and see the speedup on real hardware.

1. **Describe the GPU memory hierarchy**: registers → shared memory (SRAM) → L2 cache → HBM (DRAM).
2. **Understand Tiling for MatMul**: How tile-based computation reduces global memory access.
3. **Implement Online Softmax**: Numerically stable, incremental softmax that avoids materializing the full $N \times N$ attention matrix.
4. **Grasp FlashAttention V1 & V2**: The key algorithmic improvements and their IO complexity.
5. **Benchmark**: Compare standard attention vs. `torch.nn.functional.scaled_dot_product_attention` (which uses FlashAttention internally).

---


## 1. Environment Setup

In [2]:
import math
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory (HBM): {props.total_memory / 1e9:.1f} GB")
    print(f"SM count: {props.multi_processor_count}")

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.6.0+gitdbfe118
GPU: AMD Radeon Graphics
GPU Memory (HBM): 12.4 GB
SM count: 8


## 2. GPU Memory Hierarchy — Why IO Matters

| Level | Name | Typical Size (A100) | Bandwidth | Latency |
|-------|------|--------------------:|----------:|--------:|
| Registers | per-thread storage | ~256 KB / SM | — | ~1 cycle |
| Shared Memory (SRAM) | on-chip, per-SM | 164 KB / SM (configurable) | ~19 TB/s | ~30 cycles |
| L2 Cache | on-chip, shared | 40 MB | ~5 TB/s | ~200 cycles |
| HBM (Global Memory) | off-chip DRAM | 80 GB | 2.0 TB/s | ~400+ cycles |

**Key insight**: Shared memory is **~10×** faster than HBM. If we can keep intermediate results (attention scores, softmax statistics) in SRAM instead of writing them back to HBM, we save massive amounts of IO.

### Standard Attention IO Analysis

For a sequence of length $N$ and head dimension $d$:

1. Read Q, K from HBM → compute $S = QK^T$ → write $S$ to HBM  
2. Read $S$ → compute $P = \text{softmax}(S)$ → write $P$ to HBM  
3. Read $P$, V → compute $O = PV$ → write $O$ to HBM  

**Total HBM access**: $O(Nd + N^2)$ — dominated by the $N \times N$ matrices $S$ and $P$.

In [3]:
# Standard attention — materializes the full N×N score matrix
def standard_attention(Q, K, V):
    """Naive scaled dot-product attention.
    Args: Q, K, V — (batch, num_heads, seq_len, head_dim)
    Returns: output — same shape as V
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, N, N)  ← stored in HBM
    attn_weights = torch.softmax(scores, dim=-1)                     # (B, H, N, N)  ← stored in HBM
    output = torch.matmul(attn_weights, V)                           # (B, H, N, d)
    return output

# Quick shape check
B, H, N, D = 1, 4, 8, 16
Q = torch.randn(B, H, N, D, device=device)
K = torch.randn(B, H, N, D, device=device)
V = torch.randn(B, H, N, D, device=device)
out = standard_attention(Q, K, V)
print(f"Standard attention output shape: {out.shape}")
print(f"Score matrix size: ({N}×{N}) = {N*N} elements per head")
print(f"  → For N=4096: {4096*4096:,} elements = {4096*4096*2/1e6:.1f} MB (FP16) per head")

Standard attention output shape: torch.Size([1, 4, 8, 16])
Score matrix size: (8×8) = 64 elements per head
  → For N=4096: 16,777,216 elements = 33.6 MB (FP16) per head


## 3. Tiling for Matrix Multiplication

**Tiling** partitions large matrices into blocks (tiles) that fit in shared memory. Each tile is loaded once and reused for multiple computations, reducing global memory reads.

### Example: 32×32 MatMul with 16×16 tiles
- Without tiling: each element of C reads 32 values of A and 32 values of B from global memory → total reads = $2 \times 32 \times 32 \times 32 = 65{,}536$
- With 16×16 tiles: total reads drop to $\frac{65{,}536}{16} = 4{,}096$

**Challenge for Attention**: Tiling works directly for matmul, but attention includes a **row-wise softmax** that depends on the *entire* row. We cannot compute softmax for a tile until we have seen all tiles in that row.

→ This is solved by **online softmax**.

In [4]:
# Demonstrate tiled matmul vs naive matmul (pure Python for clarity)
def naive_matmul(A, B):
    """Element-wise matmul — each C[i,j] reads a full row/col from global memory."""
    M, K_ = A.shape
    K2, N = B.shape
    C = torch.zeros(M, N, device=A.device)
    global_reads = 0
    for i in range(M):
        for j in range(N):
            for k in range(K_):
                C[i, j] += A[i, k] * B[k, j]
                global_reads += 2  # read A[i,k] and B[k,j]
    return C, global_reads

def tiled_matmul(A, B, tile_size=4):
    """Tile-based matmul — tiles are loaded into 'shared memory' once."""
    M, K_ = A.shape
    K2, N = B.shape
    C = torch.zeros(M, N, device=A.device)
    global_reads = 0
    for ti in range(0, M, tile_size):
        for tj in range(0, N, tile_size):
            # C_tile accumulates in "registers"
            for tk in range(0, K_, tile_size):
                # Load A-tile and B-tile into "shared memory" — one read each
                A_tile = A[ti:ti+tile_size, tk:tk+tile_size]  
                B_tile = B[tk:tk+tile_size, tj:tj+tile_size]
                global_reads += 2 * tile_size * tile_size
                C[ti:ti+tile_size, tj:tj+tile_size] += A_tile @ B_tile
    return C, global_reads

# Compare on an 8×8 matrix
size = 8
A = torch.randn(size, size)
B = torch.randn(size, size)

C_naive, reads_naive = naive_matmul(A, B)
C_tiled, reads_tiled = tiled_matmul(A, B, tile_size=4)

print(f"Naive  global reads: {reads_naive:,}")
print(f"Tiled  global reads: {reads_tiled:,}   (tile_size=4)")
print(f"Reduction: {reads_naive / reads_tiled:.1f}×")
print(f"Results match: {torch.allclose(C_naive, C_tiled, atol=1e-5)}")

Naive  global reads: 1,024
Tiled  global reads: 256   (tile_size=4)
Reduction: 4.0×
Results match: True


## 4. Online Softmax — The Key Enabler

Standard softmax requires two passes over the data:
1. Find $m = \max_j x_j$ (for numerical stability)
2. Compute $\sum_j e^{x_j - m}$ and normalize

**Online softmax** maintains running statistics $(m_i, d_i)$ so that each new block of values can be incorporated in a single pass:

$$m_{\text{new}} = \max(m_{\text{old}}, m_{\text{block}})$$
$$d_{\text{new}} = d_{\text{old}} \cdot e^{m_{\text{old}} - m_{\text{new}}} + \sum_{j \in \text{block}} e^{x_j - m_{\text{new}}}$$

This is what allows FlashAttention to compute softmax **tile by tile** without materializing the full $N \times N$ matrix.

In [5]:
def online_softmax(x: torch.Tensor, block_size: int = 4) -> torch.Tensor:
    """Compute softmax of 1-D tensor x using online (incremental) algorithm.

    Processes x in blocks of `block_size`, maintaining running max and sum.
    """
    N = x.shape[0]
    m = torch.tensor(float("-inf"))  # running max
    d = torch.tensor(0.0)            # running sum of exp

    for start in range(0, N, block_size):
        block = x[start : start + block_size]
        m_block = block.max()
        m_new = torch.max(m, m_block)
        # Re-scale old sum and add new block contribution
        d = d * torch.exp(m - m_new) + torch.exp(block - m_new).sum()
        m = m_new

    # Final softmax values
    return torch.exp(x - m) / d


# Verify against PyTorch softmax
x = torch.randn(16)
result_online = online_softmax(x, block_size=4)
result_torch  = torch.softmax(x, dim=0)

print(f"Online softmax: {result_online[:6].tolist()}")
print(f"Torch softmax:  {result_torch[:6].tolist()}")
print(f"Max absolute error: {(result_online - result_torch).abs().max().item():.2e}")
print(f"Match: {torch.allclose(result_online, result_torch, atol=1e-6)}  ✓")

Online softmax: [0.21128815412521362, 0.08426158875226974, 0.007283822633326054, 0.009896711446344852, 0.02673644758760929, 0.15719261765480042]
Torch softmax:  [0.21128815412521362, 0.08426159620285034, 0.007283822633326054, 0.009896711446344852, 0.02673644758760929, 0.15719260275363922]
Max absolute error: 1.49e-08
Match: True  ✓


## 5. FlashAttention Algorithm

### FlashAttention V1 — Core Idea

Instead of computing the full $N \times N$ attention matrix, FlashAttention processes Q, K, V in **tiles**:

1. Divide K, V into blocks of $B_c$ rows, and Q into blocks of $B_r$ rows
2. For each Q-block, iterate over K/V-blocks:
   - Load Q-tile, K-tile, V-tile into shared memory (SRAM)
   - Compute local $S_{\text{tile}} = Q_{\text{tile}} K_{\text{tile}}^T$
   - Update running softmax statistics (online softmax)
   - Accumulate output: $O \mathrel{{+}{=}} \text{softmax\_tile} \times V_{\text{tile}}$
3. After all K/V-blocks are processed, apply final rescaling

**HBM access**: $O(N^2 d^2 / M)$ where $M$ is SRAM size — much less than $O(Nd + N^2)$.

### FlashAttention V2 — Improvements

1. **Swap inner/outer loop**: V1 loops K,V (outer) × Q (inner); V2 loops Q (outer) × K,V (inner), avoiding redundant writes to O
2. **Defer rescaling**: V1 rescales O every iteration; V2 only rescales once at the end
3. **Better warp partitioning**: V2 splits Q across warps (not K), eliminating inter-warp communication

In [6]:
def flash_attention_sim(Q, K, V, block_size=4):
    """Simplified FlashAttention V2 (Python simulation).

    This mirrors the real algorithm's logic but runs on CPU/GPU tensors
    without custom CUDA kernels. The purpose is educational.

    Args:
        Q, K, V: (seq_len, head_dim)
        block_size: tile size for K/V blocks
    Returns:
        O: (seq_len, head_dim)
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    O = torch.zeros_like(Q)
    l = torch.zeros(N, 1, device=Q.device)   # running sum of exp (denominator)
    m = torch.full((N, 1), float("-inf"), device=Q.device)  # running max

    # Outer loop over Q blocks (V2 style)
    for j in range(0, N, block_size):
        Kj = K[j : j + block_size]  # (block_size, d)  — loaded to SRAM
        Vj = V[j : j + block_size]  # (block_size, d)

        # Compute local attention scores for ALL query positions vs. this K-block
        S_block = (Q @ Kj.T) * scale  # (N, block_size)

        # Online softmax update
        m_block = S_block.max(dim=-1, keepdim=True).values  # (N, 1)
        m_new   = torch.max(m, m_block)

        # Correction factor for previously accumulated values
        correction = torch.exp(m - m_new)

        # New exponentials for this block
        p_block = torch.exp(S_block - m_new)  # (N, block_size)

        # Update running statistics
        l_new = l * correction + p_block.sum(dim=-1, keepdim=True)

        # Update output accumulator
        O = O * correction + p_block @ Vj  # rescale old O and add new contribution

        m = m_new
        l = l_new

    # Final normalization
    O = O / l
    return O


# Verify against standard attention
N, d = 16, 8
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

out_standard = standard_attention(
    Q.unsqueeze(0).unsqueeze(0),
    K.unsqueeze(0).unsqueeze(0),
    V.unsqueeze(0).unsqueeze(0),
).squeeze()

out_flash = flash_attention_sim(Q, K, V, block_size=4)

print(f"Standard output shape: {out_standard.shape}")
print(f"Flash sim output shape: {out_flash.shape}")
print(f"Max error: {(out_standard - out_flash).abs().max().item():.2e}")
print(f"Results match: {torch.allclose(out_standard, out_flash, atol=1e-5)}  ✓")

Standard output shape: torch.Size([16, 8])
Flash sim output shape: torch.Size([16, 8])
Max error: 2.38e-07
Results match: True  ✓


## 6. Benchmarking: Standard vs. Flash Attention

PyTorch ≥ 2.0 provides `torch.nn.functional.scaled_dot_product_attention` (SDPA) which automatically selects the most efficient backend: **FlashAttention**, **Memory-Efficient Attention**, or the **Math** fallback.

We compare wall-clock time and peak memory for increasing sequence lengths.

In [7]:
def benchmark_attention(seq_lengths, batch=1, heads=8, head_dim=64, warmup=3, repeats=10):
    """Benchmark standard vs SDPA (FlashAttention backend) attention."""
    results = []

    for N in seq_lengths:
        Q = torch.randn(batch, heads, N, head_dim, device=device, dtype=torch.float16)
        K = torch.randn(batch, heads, N, head_dim, device=device, dtype=torch.float16)
        V = torch.randn(batch, heads, N, head_dim, device=device, dtype=torch.float16)

        # --- Standard attention ---
        torch.cuda.synchronize() if device.type == "cuda" else None
        for _ in range(warmup):
            _ = standard_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None

        torch.cuda.reset_peak_memory_stats() if device.type == "cuda" else None
        t0 = time.perf_counter()
        for _ in range(repeats):
            _ = standard_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None
        std_time = (time.perf_counter() - t0) / repeats * 1000  # ms
        std_mem = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else 0

        # --- SDPA (FlashAttention backend) ---
        for _ in range(warmup):
            _ = F.scaled_dot_product_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None

        torch.cuda.reset_peak_memory_stats() if device.type == "cuda" else None
        t0 = time.perf_counter()
        for _ in range(repeats):
            _ = F.scaled_dot_product_attention(Q, K, V)
        torch.cuda.synchronize() if device.type == "cuda" else None
        sdpa_time = (time.perf_counter() - t0) / repeats * 1000
        sdpa_mem = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else 0

        speedup = std_time / sdpa_time if sdpa_time > 0 else float("inf")
        results.append((N, std_time, sdpa_time, speedup, std_mem, sdpa_mem))

        print(f"N={N:>5d} | Standard: {std_time:7.2f} ms, {std_mem:8.1f} MB | "
              f"SDPA: {sdpa_time:7.2f} ms, {sdpa_mem:8.1f} MB | "
              f"Speedup: {speedup:.2f}×")

    return results


if device.type == "cuda":
    print("=== Attention Benchmark ===")
    seq_lengths = [256, 512, 1024, 2048, 4096]
    results = benchmark_attention(seq_lengths)
else:
    print("GPU not available — benchmark skipped.")
    print("The SDPA function still works on CPU but without FlashAttention speedup.")
    # Quick correctness check on CPU
    N, H, D = 32, 2, 16
    Q = torch.randn(1, H, N, D)
    K = torch.randn(1, H, N, D)
    V = torch.randn(1, H, N, D)
    out_std  = standard_attention(Q, K, V)
    out_sdpa = F.scaled_dot_product_attention(Q, K, V)
    print(f"CPU correctness check: match={torch.allclose(out_std, out_sdpa, atol=1e-5)}")

=== Attention Benchmark ===
N=  256 | Standard:    0.11 ms,     36.7 MB | SDPA:    0.04 ms,     34.6 MB | Speedup: 2.85×
N=  512 | Standard:    0.49 ms,     44.0 MB | SDPA:    0.17 ms,     35.7 MB | Speedup: 2.97×
N= 1024 | Standard:    1.54 ms,     71.3 MB | SDPA:    0.52 ms,     37.8 MB | Speedup: 2.98×
N= 2048 | Standard:    5.91 ms,    176.2 MB | SDPA:    1.76 ms,     42.0 MB | Speedup: 3.36×
N= 4096 | Standard:   22.22 ms,    587.2 MB | SDPA:    6.79 ms,     50.5 MB | Speedup: 3.27×


## 7. IO Complexity Comparison

| Method | HBM Reads/Writes | Peak HBM for intermediates |
|--------|----------------:|:--------------------------:|
| Standard Attention | $O(Nd + N^2)$ | $O(N^2)$ (stores full score matrix) |
| FlashAttention V1 | $O(N^2 d^2 / M)$ | $O(N)$ (only stores row-wise stats) |
| FlashAttention V2 | Same asymptotic, fewer non-matmul ops, better warp utilization | $O(N)$ |

Where $M$ = SRAM size per SM (typically 100–200 KB), $N$ = sequence length, $d$ = head dimension.

For typical settings ($d=64{-}128$, $M \gg d^2$), FlashAttention eliminates almost all $N^2$-sized HBM traffic.

In [8]:
# Calculate theoretical memory savings
def memory_analysis(N, d, dtype_bytes=2):
    """Compare memory footprint of standard vs flash attention."""
    # Standard: stores S (N×N) and P (N×N)
    std_intermediate = 2 * N * N * dtype_bytes  # bytes
    # Flash: only stores running max, sum, and output per row
    flash_intermediate = N * (1 + 1 + d) * dtype_bytes  # m, l, O per row
    return std_intermediate, flash_intermediate

print("=== Intermediate Memory Comparison (per head, FP16) ===")
print(f"{'N':>6s} | {'Standard':>12s} | {'Flash':>12s} | {'Savings':>8s}")
print("-" * 48)
for N in [512, 1024, 2048, 4096, 8192, 16384]:
    std_mem, flash_mem = memory_analysis(N, d=128)
    print(f"{N:>6d} | {std_mem/1e6:>10.2f} MB | {flash_mem/1e6:>10.4f} MB | "
          f"{std_mem/flash_mem:>6.0f}×")

=== Intermediate Memory Comparison (per head, FP16) ===
     N |     Standard |        Flash |  Savings
------------------------------------------------
   512 |       1.05 MB |     0.1331 MB |      8×
  1024 |       4.19 MB |     0.2662 MB |     16×
  2048 |      16.78 MB |     0.5325 MB |     32×
  4096 |      67.11 MB |     1.0650 MB |     63×
  8192 |     268.44 MB |     2.1299 MB |    126×
 16384 |    1073.74 MB |     4.2598 MB |    252×


## Conclusions

### Technical Concepts Learned
- **GPU Memory Hierarchy**: Registers → Shared Memory (SRAM) → L2 → HBM (DRAM); bandwidth differs by 10–100×
- **Tiling**: Partitioning matrices into tiles that fit in SRAM to reduce global memory round-trips
- **Online Softmax**: Incrementally computing softmax with running max and sum, enabling tile-by-tile attention
- **FlashAttention V1**: Tiled attention with online softmax, reducing HBM access from $O(N^2)$ to $O(N^2 d^2/M)$
- **FlashAttention V2**: Improved loop ordering (Q-outer), deferred rescaling, better warp partitioning
- **SDPA**: PyTorch's `scaled_dot_product_attention` automatically uses FlashAttention when available

### Experiment Further
- Install `flash-attn` package and use `flash_attn.flash_attn_func` directly
- Compare FlashAttention performance across FP16 and BF16
- Benchmark with causal masks (autoregressive) vs. bidirectional attention
- Profile a Transformer block with `torch.profiler` to see real SRAM/HBM usage

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
